# Data Quality Audit

This notebook runs quality checks on the atlas data and reports statistics
useful for monitoring data health over time.

In [ ]:
import yaml
import json
from pathlib import Path
from collections import Counter

ROOT = Path('..')
PROBLEMS_DIR = ROOT / 'data' / 'problems'
LEADS_DIR = ROOT / 'data' / 'leads'
ATTEMPTS_DIR = ROOT / 'data' / 'attempts'
COLLECTIONS_DIR = ROOT / 'data' / 'collections'
SCHEMA_DIR = ROOT / 'schema'

In [ ]:
# Load all data
def load_yamls(directory):
    items = []
    if directory.exists():
        for f in sorted(directory.rglob('*.yaml')):
            with open(f) as fp:
                data = yaml.safe_load(fp)
                if data:
                    items.append(data)
    return items

problems = load_yamls(PROBLEMS_DIR)
leads = load_yamls(LEADS_DIR)
attempts = load_yamls(ATTEMPTS_DIR)
collections = load_yamls(COLLECTIONS_DIR)

print(f'Problems:    {len(problems)}')
print(f'Leads:       {len(leads)}')
print(f'Attempts:    {len(attempts)}')
print(f'Collections: {len(collections)}')

In [ ]:
# Check: all problems have required fields
required_fields = ['id', 'title', 'status', 'domain', 'subdomains', 'statement', 'sources']
missing = {}

for p in problems:
    pid = p.get('id', 'UNKNOWN')
    for field in required_fields:
        if field not in p:
            missing.setdefault(field, []).append(pid)

if missing:
    print('Missing required fields:')
    for field, pids in missing.items():
        print(f'  {field}: {len(pids)} problems')
else:
    print('All problems have required fields. ✓')

In [ ]:
# Check: all problems have canonical source
no_source = []
no_evidence = []

for p in problems:
    sources = p.get('sources', {})
    if not sources.get('canonical'):
        no_source.append(p.get('id', '?'))
    if not sources.get('status_evidence'):
        no_evidence.append(p.get('id', '?'))

print(f'Problems without canonical source: {len(no_source)}')
if no_source:
    for pid in no_source[:5]:
        print(f'  {pid}')

print(f'Problems without status evidence: {len(no_evidence)}')
if no_evidence:
    for pid in no_evidence[:5]:
        print(f'  {pid}')

In [ ]:
# Check: unique IDs
ids = [p.get('id') for p in problems]
dupes = [pid for pid, count in Counter(ids).items() if count > 1]

if dupes:
    print(f'Duplicate IDs found: {dupes}')
else:
    print(f'All {len(ids)} IDs are unique. ✓')

In [ ]:
# Check: collection references
problem_ids = set(ids)
bad_refs = []

for col in collections:
    for entry in col.get('problems', []):
        pid = entry.get('problem_id')
        if pid and pid not in problem_ids:
            bad_refs.append((col.get('collection_id', '?'), pid))

if bad_refs:
    print(f'Broken collection references: {len(bad_refs)}')
    for col_id, pid in bad_refs[:5]:
        print(f'  {col_id} -> {pid}')
else:
    print('All collection references valid. ✓')

In [ ]:
# Score completeness
score_fields = ['impact', 'underexplored', 'toolability', 'formality', 'ai_fit']
incomplete_scores = []

for p in problems:
    scores = p.get('scores', {})
    missing_scores = [s for s in score_fields if s not in scores]
    if missing_scores:
        incomplete_scores.append((p.get('id', '?'), missing_scores))

print(f'Problems with incomplete scores: {len(incomplete_scores)}/{len(problems)}')
if incomplete_scores:
    for pid, ms in incomplete_scores[:3]:
        print(f'  {pid}: missing {ms}')

In [ ]:
# Summary report
print('\n' + '='*60)
print('DATA QUALITY SUMMARY')
print('='*60)
print(f'Total problems:            {len(problems)}')
print(f'Open:                      {sum(1 for p in problems if p.get("status",{}).get("label") == "open")}')
print(f'Solved:                    {sum(1 for p in problems if p.get("status",{}).get("label") == "solved")}')
print(f'With canonical source:     {len(problems) - len(no_source)}')
print(f'With status evidence:      {len(problems) - len(no_evidence)}')
print(f'With complete scores:      {len(problems) - len(incomplete_scores)}')
print(f'Unique IDs:                {"✓" if not dupes else "✗ DUPLICATES"}')
print(f'Collection refs valid:     {"✓" if not bad_refs else "✗ BROKEN"}')
print(f'Leads in radar:            {len(leads)}')
print(f'Attempts recorded:         {len(attempts)}')
print(f'Collections curated:       {len(collections)}')